In [1]:
import rasterio as rio
import os
from pathlib import Path
import numpy as np
from glob import glob

In [2]:
path = '/home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/monthly_lst/'
output_path = '/home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/'
naming_convention = 'GEMLST_monthly_YYYY_MM.tif'

files = sorted(glob(os.path.join(path, 'GEMLST_monthly_*.tif')))

# Remove Jan, Feb 2000 and Dec 2025
files_to_remove = [
    os.path.join(path, 'GEMLST_monthly_2000_01.tif'),
    os.path.join(path, 'GEMLST_monthly_2000_02.tif'),
    os.path.join(path, 'GEMLST_monthly_2025_12.tif')
]
files = [f for f in files if f not in files_to_remove]

In [3]:
 # Open the first file to get metadata and shape
with rio.open(files[0]) as src:
    profile = src.profile
    height, width = src.height, src.width
    dtype = src.dtypes[0]

# Get year and month from filename
def get_year_month(filename):
    basename = os.path.basename(filename)
    parts = basename.split('_')
    year = parts[2]
    month = parts[3][0:2]
    return year, month

# Iterate in steps of 3 months
for i in range(0, len(files), 3):
    group = files[i:i+3]
    if len(group) != 3:
        print(f"Skipping incomplete group: {group}")
        continue

    # Get year and month for the second file in the group
    year, month = get_year_month(group[1])

    # Determine season based on the second month in the group

    if month == '01':
        season = 'DJF'
    elif month == '04':
        season = 'MAM'
    elif month == '07':
        season = 'JJA'
    elif month == '10':
        season = 'SON'

    else:
        print(f"Unexpected month {month} for group starting at {group[0]}")
        continue


    stack = np.zeros((3, height, width), dtype=dtype)

    # Read all files in the group
    for j, fname in enumerate(group):
        with rio.open(fname) as src:
            stack[j] = src.read(1)  # Assuming single-band images

    # Compute the average
    avg = np.mean(stack, axis=0)

    # Save the average image with the correct naming
    output_dir = os.path.join(output_path, f'GEMLST_{year}_{season}.tif')
    with rio.open(
        output_dir,
        'w',
        driver='GTiff',
        height=height,
        width=width,
        count=1,
        dtype=avg.dtype,
        crs=profile['crs'],
        transform=profile['transform']
    ) as dst:
        dst.write(avg, 1)

    print(f"Saved {output_dir} with 3 images averaged.")

print("Done!")

Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2000_MAM.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2000_JJA.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2000_SON.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2001_DJF.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2001_MAM.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2001_JJA.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2001_SON.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quarterly_lst/GEMLST_2002_DJF.tif with 3 images averaged.
Saved /home/jovyan/work/AVOCA/GEM/Development/Anlaysis/TSA/quart